# Oregon PM2.5 Air Quality Analysis
**Author:** Lexie Smith  
**Data Source:** EPA Air Quality System (AQS) — Daily PM2.5 FRM/FEM Mass (Parameter 88101)  
**Data Range:** 1999–2022 (Oregon monitoring stations only)  
**Tools:** Python, MySQL, Power BI

## Project Overview
This notebook documents the data pipeline for an analysis of PM2.5 air quality trends across Oregon counties from 1999 to 2022. The pipeline covers:
1. Automated bulk download of 35 years of EPA AQS daily PM2.5 data
2. File extraction and organization
3. Data exploration and column selection
4. Loading filtered Oregon data into a normalized MySQL database

SQL analysis queries and Power BI dashboard are maintained separately (see repository).

## Research Questions
- Has the number of days exceeding the federal PM2.5 standard (35 µg/m³) changed over time in Oregon?
- Which counties show the highest PM2.5 exposure when normalized by monitoring station count?
- Are unhealthy air days concentrated in specific seasons, and what does that suggest about pollution sources?

## Step 1: Download EPA AQS Daily PM2.5 Data

The EPA Air Quality System (AQS) publishes annual CSV files for each criteria pollutant at:
https://aqs.epa.gov/aqsweb/airdata/download_files.html

Parameter code `88101` corresponds to PM2.5 FRM/FEM Mass which is the regulatory standard measurement.  
Files are downloaded programmatically for all available years (1990–2024) and saved as zip archives.

**Note:** Early years (1990–1998) contain no Oregon data. PM2.5 became a regulated pollutant under the NAAQS in 1997, so widespread state monitoring did not begin until 1999.

In [1]:
import requests
import os

# Set your local data directory
save_folder = r"C:\Users\lexie\Documents\DEQ_project\data\pm25_daily"
os.makedirs(save_folder, exist_ok=True)

# EPA AQS bulk download URL pattern for daily PM2.5
base_url = "https://aqs.epa.gov/aqsweb/airdata/daily_88101_{year}.zip"

# PM2.5 data goes back to 1990 on AQS
years = range(1990, 2025)

for year in years:
    url = base_url.format(year=year)
    filename = os.path.join(save_folder, f"daily_pm25_{year}.zip")
    
    print(f"Downloading {year}...")
    response = requests.get(url)
    
    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)
        print(f"  Saved {year}")
    else:
        print(f"  Skipped {year} (not available)")

print("Done!")

  Saved 1990
  Saved 1991
  Saved 1992


KeyboardInterrupt: 

## Step 2: Extract and Organize Files

Downloaded zip files are extracted to a separate folder.  
Some years (2020–2023) unzipped into subfolders rather than directly, so these are moved to the flat directory structure for consistent processing.

In [ ]:
import zipfile

unzip_folder = "C:/Users/lexie/Documents/DEQ_project/data/pm25_unzipped"
os.makedirs(unzip_folder, exist_ok=True)

for filename in os.listdir(save_folder):
    if filename.endswith(".zip"):
        zip_path = os.path.join(save_folder, filename)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(unzip_folder)
        print(f"Unzipped {filename}")

print("All done!")

Unzipped daily_pm25_1990.zip
Unzipped daily_pm25_1991.zip
Unzipped daily_pm25_1992.zip
Unzipped daily_pm25_1993.zip
Unzipped daily_pm25_1994.zip
Unzipped daily_pm25_1995.zip
Unzipped daily_pm25_1996.zip
Unzipped daily_pm25_1997.zip
Unzipped daily_pm25_1998.zip
Unzipped daily_pm25_1999.zip
Unzipped daily_pm25_2000.zip
Unzipped daily_pm25_2001.zip
Unzipped daily_pm25_2002.zip
Unzipped daily_pm25_2003.zip
Unzipped daily_pm25_2004.zip
Unzipped daily_pm25_2005.zip
Unzipped daily_pm25_2006.zip
Unzipped daily_pm25_2007.zip
Unzipped daily_pm25_2008.zip
Unzipped daily_pm25_2009.zip
Unzipped daily_pm25_2010.zip
Unzipped daily_pm25_2011.zip
Unzipped daily_pm25_2012.zip
Unzipped daily_pm25_2013.zip
Unzipped daily_pm25_2014.zip
Unzipped daily_pm25_2015.zip
Unzipped daily_pm25_2016.zip
Unzipped daily_pm25_2017.zip
Unzipped daily_pm25_2018.zip
Unzipped daily_pm25_2019.zip
Unzipped daily_pm25_2020.zip
Unzipped daily_pm25_2021.zip
Unzipped daily_pm25_2022.zip
Unzipped daily_pm25_2023.zip
Unzipped daily

In [ ]:
import shutil

problem_years = ["daily_88101_2020", "daily_88101_2021", "daily_88101_2022", "daily_88101_2023"]

for folder in problem_years:
    folder_path = os.path.join(unzip_folder, folder)
    csv_name = folder + ".csv"
    src = os.path.join(folder_path, csv_name)
    dst = os.path.join(unzip_folder, csv_name)
    shutil.move(src, dst)
    print(f"Moved {csv_name}")

print("Done!")

Moved daily_88101_2020.csv
Moved daily_88101_2021.csv
Moved daily_88101_2022.csv
Moved daily_88101_2023.csv
Done!


In [ ]:
for i, f in enumerate(os.listdir(unzip_folder)):
    print(i, f)

0 daily_88101_1990.csv
1 daily_88101_1991.csv
2 daily_88101_1992.csv
3 daily_88101_1993.csv
4 daily_88101_1994.csv
5 daily_88101_1995.csv
6 daily_88101_1996.csv
7 daily_88101_1997.csv
8 daily_88101_1998.csv
9 daily_88101_1999.csv
10 daily_88101_2000.csv
11 daily_88101_2001.csv
12 daily_88101_2002.csv
13 daily_88101_2003.csv
14 daily_88101_2004.csv
15 daily_88101_2005.csv
16 daily_88101_2006.csv
17 daily_88101_2007.csv
18 daily_88101_2008.csv
19 daily_88101_2009.csv
20 daily_88101_2010.csv
21 daily_88101_2011.csv
22 daily_88101_2012.csv
23 daily_88101_2013.csv
24 daily_88101_2014.csv
25 daily_88101_2015.csv
26 daily_88101_2016.csv
27 daily_88101_2017.csv
28 daily_88101_2018.csv
29 daily_88101_2019.csv
30 daily_88101_2020
31 daily_88101_2020.csv
32 daily_88101_2021
33 daily_88101_2021.csv
34 daily_88101_2022
35 daily_88101_2022.csv
36 daily_88101_2023
37 daily_88101_2023.csv
38 daily_88101_2024.csv


In [ ]:
import os
import shutil

# Remove the empty folders
problem_years = ["daily_88101_2020", "daily_88101_2021", "daily_88101_2022", "daily_88101_2023"]

for folder in problem_years:
    folder_path = os.path.join(unzip_folder, folder)
    shutil.rmtree(folder_path)
    print(f"Removed {folder}")

# Now peek at 2020
df = pd.read_csv(os.path.join(unzip_folder, "daily_88101_2020.csv"))
print(df.shape)
print(df.columns.tolist())
print(df.head(3))

Removed daily_88101_2020
Removed daily_88101_2021
Removed daily_88101_2022
Removed daily_88101_2023
(704091, 29)
['State Code', 'County Code', 'Site Num', 'Parameter Code', 'POC', 'Latitude', 'Longitude', 'Datum', 'Parameter Name', 'Sample Duration', 'Pollutant Standard', 'Date Local', 'Units of Measure', 'Event Type', 'Observation Count', 'Observation Percent', 'Arithmetic Mean', '1st Max Value', '1st Max Hour', 'AQI', 'Method Code', 'Method Name', 'Local Site Name', 'Address', 'State Name', 'County Name', 'City Name', 'CBSA Name', 'Date of Last Change']
   State Code  County Code  Site Num  Parameter Code  POC   Latitude  \
0           1            3        10           88101    1  30.497478   
1           1            3        10           88101    1  30.497478   
2           1            3        10           88101    1  30.497478   

   Longitude  Datum            Parameter Name Sample Duration  ...   AQI  \
0 -87.880258  NAD83  PM2.5 - Local Conditions         24 HOUR  ...  56.0 

In [ ]:
print(df["Event Type"].unique())
print(df["Pollutant Standard"].unique())
print(df["Sample Duration"].unique())

[nan 'Included' 'Excluded']
['PM25 24-hour 2012' nan]
['24 HOUR' '1 HOUR' '24-HR BLK AVG']


In [ ]:
import subprocess
subprocess.run(["pip", "install", "mysql-connector-python"])

CompletedProcess(args=['pip', 'install', 'mysql-connector-python'], returncode=0)

## Step 4: Load Data into MySQL

Data is loaded into a normalized MySQL database (`oregon_air_quality`) with two tables:
- `monitors` — one row per monitoring station (location, county, coordinates)
- `daily_readings` — one row per day per monitor (PM2.5 values, AQI, event type)

**Database credentials are stored as environment variables and not hardcoded.**  
Set the following environment variables before running:
- `MYSQL_USER`
- `MYSQL_PASSWORD`

**Data quality handling:**
- Missing values (`NaN`) are converted to `NULL` before insertion
- Date formats vary across years — a parser handles both `YYYY-MM-DD` and `M/D/YYYY` formats
- Duplicate monitor entries are skipped using `INSERT IGNORE`

In [ ]:
import pandas as pd
import mysql.connector
import os
from datetime import datetime  # NEW

def clean(val):
    if pd.isna(val):
        return None
    return val

def clean_date(val):  # NEW
    if pd.isna(val):
        return None
    try:
        return datetime.strptime(str(val), "%Y-%m-%d").date()
    except ValueError:
        return datetime.strptime(str(val), "%m/%d/%Y").date()

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="1234",
    database="oregon_air_quality",
    connection_timeout=300
)
cursor = conn.cursor()

unzip_folder = r"C:\Users\lexie\Documents\DEQ_project\data\pm25_unzipped"
inserted_monitors = set()

for filename in sorted(os.listdir(unzip_folder)):
    if not filename.endswith(".csv"):
        continue
        
    year = filename.split("_")[2].split(".")[0]
    
    if int(year) < 2020:  # NEW - skip already loaded years
        print(f"Skipping {year}")
        continue
    
    print(f"Processing {year}...")
    
    df = pd.read_csv(os.path.join(unzip_folder, filename))
    df = df[df["State Name"] == "Oregon"]
    df = df[df["Sample Duration"] == "24 HOUR"]
    
    df["site_id"] = (
        df["State Code"].astype(str).str.zfill(2) + "-" +
        df["County Code"].astype(str).str.zfill(3) + "-" +
        df["Site Num"].astype(str).str.zfill(4)
    )
    
    for _, row in df.drop_duplicates("site_id").iterrows():
        if row["site_id"] not in inserted_monitors:
            cursor.execute("""
                INSERT IGNORE INTO monitors 
                (site_id, state_code, county_code, site_num,
                 state_name, county_name, city_name, latitude, longitude)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
            """, (
                clean(row["site_id"]), clean(row["State Code"]), clean(row["County Code"]),
                clean(row["Site Num"]), clean(row["State Name"]), clean(row["County Name"]),
                clean(row["City Name"]), clean(row["Latitude"]), clean(row["Longitude"])
            ))
            inserted_monitors.add(row["site_id"])
    
    for _, row in df.iterrows():
        cursor.execute("""
            INSERT INTO daily_readings
            (site_id, date_local, event_type, observation_count,
             observation_percent, arithmetic_mean, max_value, aqi)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        """, (
            clean(row["site_id"]),
            clean_date(row["Date Local"]),  # CHANGED
            clean(row["Event Type"]),
            clean(row["Observation Count"]),
            clean(row["Observation Percent"]),
            clean(row["Arithmetic Mean"]),
            clean(row["1st Max Value"]),
            clean(row["AQI"])
        ))
    
    conn.commit()
    print(f"  Done {year}")

print("All data loaded!")
cursor.close()
conn.close()

Skipping 1990
Skipping 1991
Skipping 1992
Skipping 1993
Skipping 1994
Skipping 1995
Skipping 1996
Skipping 1997
Skipping 1998
Skipping 1999
Skipping 2000
Skipping 2001
Skipping 2002
Skipping 2003
Skipping 2004
Skipping 2005
Skipping 2006
Skipping 2007
Skipping 2008
Skipping 2009
Skipping 2010
Skipping 2011
Skipping 2012
Skipping 2013
Skipping 2014
Skipping 2015
Skipping 2016
Skipping 2017
Skipping 2018
Skipping 2019
Processing 2020...
  Done 2020
Processing 2021...
  Done 2021
Processing 2022...
  Done 2022
Processing 2023...
  Done 2023
Processing 2024...


C:\Users\lexie\AppData\Local\Temp\ipykernel_40824\983808903.py:43: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(unzip_folder, filename))


  Done 2024
All data loaded!
